In [0]:
#%pip install databricks-zerobus-ingest-sdk

In [0]:
#dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("login", "mariiasirska")
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("region", "eastus")   
dbutils.widgets.text("secret_scope", "scope_mariiasirska") 
dbutils.widgets.text("sp_application_id", "1df43f4a-73ed-430b-a492-39c410219a0a")
dbutils.widgets.text("n_orders", "500")
dbutils.widgets.text("checkpoint_path", "") 

In [0]:
login = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("catalog")
bronze = f"{catalog}.{login}_bronze"
silver = f"{catalog}.{login}_silver"

In [0]:
EVENTS_BRONZE = f"{bronze}.olist_order_events_bronze"
EVENTS_SILVER = f"{silver}.olist_order_events_silver"
CHECKPOINT = dbutils.widgets.get("checkpoint_path") or \
    f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/olist/_checkpoints/order_events_silver"
print(EVENTS_BRONZE)
print(EVENTS_SILVER)

Target table 

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EVENTS_BRONZE} (
  event_id        STRING  COMMENT 'Deterministic id: uuid5(order_id:event_type). The idempotency key',
  event_type      STRING,
  event_time      STRING  COMMENT 'ISO-8601, UTC',
  order_id        STRING,
  customer_state  STRING,
  product_category STRING,
  price           DOUBLE,
  freight_value   DOUBLE,
  producer_id     STRING  COMMENT 'Identifies one producer run (to tell replays apart)',
  seq             BIGINT  COMMENT 'Sequence number inside one producer run'
)
COMMENT 'Order events pushed directly by Zerobus Ingest (at-least-once, duplicates possible)'
""")
display(spark.sql(f"DESCRIBE DETAIL {EVENTS_BRONZE}").select("name", "format", "location"))


In [0]:
sp = dbutils.widgets.get("sp_application_id").strip()
if not sp:
    print("sp_application_id is empty: skipping GRANTs")
else:
    for stmt in [
        f"GRANT USE CATALOG ON CATALOG {catalog} TO `{sp}`",
        f"GRANT USE SCHEMA ON SCHEMA {bronze} TO `{sp}`",
        f"GRANT MODIFY, SELECT ON TABLE {EVENTS_BRONZE} TO `{sp}`",
    ]:
        try:
            spark.sql(stmt)
            print("ok:", stmt)
        except Exception as e:
            print("FAILED (ask an admin):", stmt, "->", str(e)[:150])

In [0]:
scope = dbutils.widgets.get("secret_scope").strip()
try:
    client_id = dbutils.secrets.get(scope, "zerobus-client-id-m")
    client_secret = dbutils.secrets.get(scope, "zerobus-client-secret-m")
except Exception as e:
    raise RuntimeError(f"Cannot read zerobus-client-id / zerobus-client-secret from scope '{scope}': {e}")

region = dbutils.widgets.get("region").strip()
assert region, "Fill the 'region' widget (e.g. westeurope): see the workspace switcher or the account console"

workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
workspace_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().workspaceId().get()
server_endpoint = f"https://{workspace_id}.zerobus.{region}.azuredatabricks.net"
print("workspace:", workspace_url)
print("zerobus endpoint:", server_endpoint)

 Producer:

In [0]:
import uuid, random, time, datetime as dt
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import TableProperties

In [0]:
NS = uuid.UUID("0b8f6a52-6c6e-4d6f-9f6e-5c7a4f1d2e11")
EVENT_TYPES = ["order_created", "order_shipped", "order_delivered"]
BASE_TIME = dt.datetime(2026, 11, 20, tzinfo=dt.timezone.utc)

In [0]:
def make_event_id(order_id, event_type):
    return str(uuid.uuid5(NS, f"{order_id}:{event_type}"))

In [0]:
def load_source_orders(n):
    return spark.sql(f"""
        SELECT o.order_id,
               c.state AS customer_state,
               coalesce(p.category_name_en, p.category_name, 'unknown') AS product_category,
               i.price, i.freight_value
        FROM {silver}.olist_orders_silver o
        JOIN {silver}.olist_customers_silver c ON o.customer_id = c.customer_id
        JOIN {silver}.olist_order_items_silver i ON o.order_id = i.order_id AND i.order_item_id = 1
        JOIN {silver}.olist_products_silver p ON i.product_id = p.product_id
        WHERE o.order_id NOT LIKE 'new_%'
        ORDER BY rand(42)
        LIMIT {int(n)}
    """).collect()

In [0]:
def build_events(rows, producer_id, dup_rate, seed=42):
    rnd = random.Random(seed)
    events, seq = [], 0
    for r in rows:
        created = BASE_TIME + dt.timedelta(minutes=rnd.randint(0, 24 * 60))
        for k, et in enumerate(EVENT_TYPES):
            seq += 1
            events.append({
                "event_id": make_event_id(r["order_id"], et),
                "event_type": et,
                "event_time": (created + dt.timedelta(hours=8 * k)).isoformat(),
                "order_id": r["order_id"],
                "customer_state": r["customer_state"],
                "product_category": r["product_category"],
                "price": float(r["price"]),
                "freight_value": float(r["freight_value"]),
                "producer_id": producer_id,
                "seq": seq,
            })
    n_dups = int(len(events) * dup_rate)
    events += [dict(e) for e in rnd.sample(events, n_dups)]
    return events, n_dups

In [0]:
def run_producer(tag, n_orders=500, dup_rate=0.05):
    producer_id = f"nb-{tag}"
    events, n_dups = build_events(load_source_orders(n_orders), producer_id, dup_rate)

    sdk = ZerobusSdk(server_endpoint, workspace_url)
    stream = sdk.create_stream(client_id, client_secret, TableProperties(EVENTS_BRONZE))
    t0 = time.time()
    try:
        last = 0
        for ev in events:
            last = stream.ingest_record_offset(ev)
        stream.wait_for_offset(last)
    finally:
        stream.close()
    t_ack = time.time() - t0

    t1 = time.time()
    while True:
        n_visible = spark.table(EVENTS_BRONZE).filter(F.col("producer_id") == producer_id).count()
        if n_visible >= len(events) or time.time() - t1 > 120:
            break
        time.sleep(3)
    result = {
        "producer_id": producer_id,
        "sent": len(events),
        "of_which_duplicates": n_dups,
        "visible_in_table": n_visible,
        "seconds_to_durable_ack": round(t_ack, 1),
        "seconds_until_queryable": round(time.time() - t1, 1),
    }
    print(result)
    return result

In [0]:
N_ORDERS = int(dbutils.widgets.get("n_orders"))
run1 = run_producer("run1", n_orders=N_ORDERS, dup_rate=0.05)

duplicates are visible in bronze:

In [0]:
raw = spark.table(EVENTS_BRONZE)
display(raw.orderBy(F.desc("seq")).limit(10))

stats = raw.agg(F.count("*").alias("rows"), F.countDistinct("event_id").alias("distinct_event_ids")).first()
print(f"bronze rows = {stats['rows']}, distinct event_id = {stats['distinct_event_ids']}, duplicates = {stats['rows'] - stats['distinct_event_ids']}")
display(raw.groupBy("event_type").count())

Idempotent processing: streaming MERGE into silver:

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EVENTS_SILVER} (
  event_id STRING NOT NULL,
  event_type STRING,
  event_ts TIMESTAMP,
  order_id STRING,
  customer_state STRING,
  product_category STRING,
  price DECIMAL(10,2),
  freight_value DECIMAL(10,2),
  producer_id STRING,
  seq BIGINT,
  _first_seen_at TIMESTAMP
)
COMMENT 'De-duplicated order events (idempotent MERGE on event_id)'
""")

In [0]:
def upsert_events(batch_df, batch_id):
    deduped = (
        batch_df.filter(F.col("event_id").isNotNull())
        .select(
            "event_id", "event_type",
            F.to_timestamp("event_time").alias("event_ts"),
            "order_id", "customer_state", "product_category",
            F.col("price").cast("decimal(10,2)").alias("price"),
            F.col("freight_value").cast("decimal(10,2)").alias("freight_value"),
            "producer_id", "seq",
            F.current_timestamp().alias("_first_seen_at"),
        )
        .dropDuplicates(["event_id"])
    )
    (
        DeltaTable.forName(spark, EVENTS_SILVER).alias("t")
        .merge(deduped.alias("s"), "t.event_id = s.event_id")
        .whenNotMatchedInsertAll()
        .execute()
    )


def run_silver_stream():
    q = (
        spark.readStream.table(EVENTS_BRONZE)
        .writeStream.foreachBatch(upsert_events)
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()


def counts():
    b = spark.table(EVENTS_BRONZE)
    return {
        "bronze_rows": b.count(),
        "bronze_distinct_event_ids": b.select("event_id").distinct().count(),
        "silver_rows": spark.table(EVENTS_SILVER).count(),
    }

run_silver_stream()
c1 = counts()
print(c1)
assert c1["silver_rows"] == c1["bronze_distinct_event_ids"], "silver must contain exactly one row per event_id"
assert spark.table(EVENTS_SILVER).groupBy("event_id").count().filter("count > 1").count() == 0

sending the same business events again:

In [0]:
run2 = run_producer("run2", n_orders=N_ORDERS, dup_rate=0.05)
run_silver_stream()
c2 = counts()
print("before replay:", c1)
print("after replay: ", c2)

assert c2["bronze_rows"] > c1["bronze_rows"], "bronze should have grown (duplicates are kept as raw history)"
assert c2["silver_rows"] == c1["silver_rows"], "silver must be unchanged: processing is idempotent"
print("IDEMPOTENT: bronze grew by", c2["bronze_rows"] - c1["bronze_rows"], "rows, silver unchanged at", c2["silver_rows"])

re-running the stream with no new data:

In [0]:
run_silver_stream()
c3 = counts()
print(c3)
assert c3["silver_rows"] == c2["silver_rows"]
display(
    spark.sql(f"DESCRIBE HISTORY {EVENTS_SILVER}")
    .selectExpr("version", "timestamp", "operation",
                "operationMetrics['numTargetRowsInserted'] AS inserted",
                "operationMetrics['numSourceRows'] AS source_rows")
    .orderBy("version")
)

Silver result by event type and state:

In [0]:
display(spark.sql(f"""
SELECT event_type, count(*) AS events, count(DISTINCT order_id) AS orders, round(sum(price), 2) AS price_sum
FROM {EVENTS_SILVER}
GROUP BY event_type
ORDER BY event_type
"""))
print("run1:", run1)
print("run2:", run2)

Cost visibility:

In [0]:
try:
    display(spark.sql("""
    SELECT usage_date, sku_name,
           product_features.lakeflow_connect.zerobus_request_type AS request_type,
           sum(usage_quantity) AS dbus
    FROM system.billing.usage
    WHERE billing_origin_product = 'LAKEFLOW_CONNECT' AND usage_date >= date_sub(current_date(), 14)
    GROUP BY 1, 2, 3
    ORDER BY usage_date DESC
    """))
except Exception as e:
    print("No access to system.billing.usage:", str(e)[:200])